# Model Selection for MIDAS: Lag Order and Weight Function Choice

**Module 02 — Notebook 02**  
**Estimated time:** 15 minutes

## Learning Objectives

By the end of this notebook you will be able to:

1. Compute AIC and BIC for MIDAS models with varying lag order K
2. Explain why restricted MIDAS has k=4 parameters regardless of K
3. Select the lag order P using BIC and validate with expanding-window cross-validation
4. Compare Beta MIDAS, Almon MIDAS, and U-MIDAS using information criteria
5. Detect residual serial correlation with the Ljung-Box test

## Prerequisites

- Notebook 01: NLS Optimization (profile SSE, basic estimation)
- Guide 02: Model Selection Theory

## Key Insight

For restricted Beta MIDAS, the AIC/BIC parameter count k=4 regardless of how many monthly lags K you include. The SSE decreases as K grows, but the penalty stays fixed. BIC therefore selects the largest K that provides meaningful fit improvement.

In [ ]:
callout("For restricted Beta MIDAS, the AIC/BIC parameter count k=4 regardless of how many monthly lags K you include. The SSE decreases as K grows, but the pe", kind="insight")

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist, chi2, f as f_dist
from sklearn.linear_model import LinearRegression, Ridge
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

print("Libraries loaded successfully.")

In [ ]:
section_divider("1. Load Data")

In [ ]:
learning_objectives(["Compute AIC and BIC for MIDAS models with varying lag order K", "Explain why restricted MIDAS has k=4 parameters regardless of K", "Select the lag order P using BIC and validate with expanding-window cross-validation", "Compare Beta MIDAS, Almon MIDAS, and U-MIDAS using information criteria", "Detect residual serial correlation with the Ljung-Box test"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load Data

We use the same GDP and Industrial Production data as Notebook 01. The `USE_FRED` flag lets you switch between live FRED data and the offline CSV fallback.

In [ ]:
USE_FRED = False  # Set True if you have a FRED API key configured

def load_series_from_csv(series_id):
    """Load a data series from the course CSV resources."""
    # Search up to 4 directory levels for the resources folder
    import os
    candidates = [
        '../resources',
        '../../module_00_foundations/resources',
        '../../../module_00_foundations/resources',
        '../../../../module_00_foundations/resources',
    ]
    filename_map = {
        'GDPC1':   'gdp_quarterly.csv',
        'INDPRO':  'industrial_production_monthly.csv',
    }
    fname = filename_map.get(series_id)
    if fname is None:
        raise ValueError(f"No CSV mapping for series {series_id}")
    for base in candidates:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            df = pd.read_csv(path, index_col='date', parse_dates=True)
            return df.squeeze()
    raise FileNotFoundError(f"Could not find {fname} in any candidate directory")


if USE_FRED:
    from fredapi import Fred
    fred = Fred()  # Reads FRED_API_KEY from environment
    gdp_raw = fred.get_series('GDPC1', observation_start='2000-01-01')
    ip_raw  = fred.get_series('INDPRO', observation_start='1999-01-01')
    # Compute log growth rates
    gdp_growth = np.log(gdp_raw).diff().dropna() * 100
    ip_growth  = np.log(ip_raw).diff().dropna() * 100
else:
    gdp_growth = load_series_from_csv('GDPC1')
    ip_growth  = load_series_from_csv('INDPRO')

print(f"GDP growth:  {len(gdp_growth)} quarterly observations")
print(f"IP growth:   {len(ip_growth)} monthly observations")
print(f"GDP range:   {gdp_growth.index[0]} — {gdp_growth.index[-1]}")
print(f"IP range:    {ip_growth.index[0]} — {ip_growth.index[-1]}")

In [ ]:
section_divider("2. Core Functions")

## 2. Core Functions

We define the building blocks used throughout this notebook: the MIDAS data matrix, weight functions, and profile SSE.

In [ ]:
def build_midas_matrix(y_low_freq, x_high_freq, K, m=3):
    """
    Build aligned (Y, X) arrays for MIDAS regression.

    Parameters
    ----------
    y_low_freq : pd.Series with DatetimeIndex or PeriodIndex (quarterly)
    x_high_freq : pd.Series with DatetimeIndex or PeriodIndex (monthly)
    K : int — number of high-frequency lags
    m : int — frequency ratio (3 for monthly/quarterly)

    Returns
    -------
    Y : np.ndarray, shape (T,)
    X : np.ndarray, shape (T, K)
    """
    # Convert to period index for unambiguous alignment
    if hasattr(y_low_freq.index, 'to_period'):
        y_q = y_low_freq.copy()
        y_q.index = y_low_freq.index.to_period('Q')
    else:
        y_q = y_low_freq.copy()
        y_q.index = pd.PeriodIndex(y_low_freq.index, freq='Q')

    if hasattr(x_high_freq.index, 'to_period'):
        x_m = x_high_freq.copy()
        x_m.index = x_high_freq.index.to_period('M')
    else:
        x_m = x_high_freq.copy()
        x_m.index = pd.PeriodIndex(x_high_freq.index, freq='M')

    rows_Y, rows_X = [], []

    for q in y_q.index:
        # Last month of quarter q is the 0-lag (most recent)
        last_month = q.asfreq('M', how='end')
        lags = [last_month - i for i in range(K)]

        # Check all lags are available
        if any(lag not in x_m.index for lag in lags):
            continue
        # Check quarterly observation is available
        if q not in y_q.index:
            continue

        rows_Y.append(y_q[q])
        rows_X.append([x_m[lag] for lag in lags])

    Y = np.array(rows_Y)
    X = np.array(rows_X)
    return Y, X


def beta_weights(K, theta1, theta2):
    """
    Beta polynomial weights evaluated at midpoints (j+0.5)/K.
    Reversed so j=0 is the most recent lag.
    """
    if theta1 <= 0 or theta2 <= 0:
        return np.ones(K) / K
    x = (np.arange(K) + 0.5) / K
    # Evaluate at (1-x) so j=0 (x near 0) gets weight from the right tail
    raw = beta_dist.pdf(1 - x, theta1, theta2)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K


def almon_weights(K, theta1, theta2):
    """
    Almon (exponential) polynomial weights.
    log w_j = theta1*j + theta2*j^2, then normalize.
    """
    j = np.arange(K)
    log_raw = theta1 * j + theta2 * j**2
    log_raw -= log_raw.max()  # Numerical stability
    raw = np.exp(log_raw)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K


def profile_sse(theta, Y, X, weight_fn):
    """
    Profile SSE: for given theta, solve (alpha, beta) analytically via OLS.
    Returns SSE at the optimal (alpha, beta) for this theta.
    """
    t1, t2 = theta
    if t1 <= 0.01 or t2 <= 0.01:
        return 1e10

    w = weight_fn(X.shape[1], t1, t2)
    xw = X @ w

    # Demeaned OLS to solve beta analytically
    xc = xw - xw.mean()
    yc = Y - Y.mean()
    ss = np.dot(xc, xc)
    if ss < 1e-12:
        return np.sum((Y - Y.mean())**2)

    beta = np.dot(yc, xc) / ss
    alpha = Y.mean() - beta * xw.mean()
    resid = Y - alpha - beta * xw
    return np.sum(resid**2)


def estimate_midas(Y, X, weight_fn, starts=None):
    """
    Estimate MIDAS by profile NLS with multiple random starts.

    Returns dict with theta1, theta2, alpha, beta, sse, weights.
    """
    if starts is None:
        starts = [(1.0, 1.0), (1.0, 5.0), (1.5, 4.0), (2.0, 3.0), (0.5, 3.0)]

    best_sse = np.inf
    best_result = None

    for t1_0, t2_0 in starts:
        res = minimize(
            profile_sse,
            [t1_0, t2_0],
            args=(Y, X, weight_fn),
            method='Nelder-Mead',
            options={'maxiter': 20000, 'xatol': 1e-8, 'fatol': 1e-8}
        )
        if res.fun < best_sse:
            best_sse = res.fun
            best_result = res

    t1_hat, t2_hat = best_result.x
    t1_hat = max(t1_hat, 0.01)
    t2_hat = max(t2_hat, 0.01)

    w_hat = weight_fn(X.shape[1], t1_hat, t2_hat)
    xw = X @ w_hat
    xc = xw - xw.mean()
    yc = Y - Y.mean()
    beta_hat = np.dot(yc, xc) / np.dot(xc, xc)
    alpha_hat = Y.mean() - beta_hat * xw.mean()
    resid = Y - alpha_hat - beta_hat * xw

    return {
        'theta1': t1_hat, 'theta2': t2_hat,
        'alpha': alpha_hat, 'beta': beta_hat,
        'sse': best_sse,
        'weights': w_hat,
        'residuals': resid,
        'xw': xw
    }


print("Core functions defined.")

In [ ]:
section_divider("3. AIC and BIC — The Key Insight for Restricted MIDAS")

## 3. AIC and BIC — The Key Insight for Restricted MIDAS

Recall the standard information criteria:

$$\text{AIC}(K) = T \ln\!\left(\frac{\text{SSE}(K)}{T}\right) + 2k(K)$$

$$\text{BIC}(K) = T \ln\!\left(\frac{\text{SSE}(K)}{T}\right) + k(K)\ln T$$

**For restricted Beta MIDAS:** $k = 4$ regardless of $K$ (we always estimate $\alpha, \beta, \theta_1, \theta_2$).  
**For U-MIDAS:** $k = K + 1$ (intercept plus K unrestricted weights times beta, but all absorbed into K+1 coefficients).

This means that for restricted MIDAS, BIC selects the largest $K$ that provides meaningful SSE reduction — the penalty never changes.

In [ ]:
callout("for Restricted MIDAS", kind="insight")

In [ ]:
# --- Build MIDAS matrices for P = 1, 2, 3, 4, 6 quarterly lags ---
# P quarterly lags = K = P*3 monthly lags

P_values = [1, 2, 3, 4, 6]
K_values = [p * 3 for p in P_values]   # [3, 6, 9, 12, 18]

print("Building MIDAS matrices for different lag orders...")
midas_data = {}

# Use K=18 as the maximum to build the dataset once, then slice
K_max = 18
Y_base, X_base = build_midas_matrix(gdp_growth, ip_growth, K_max)
T = len(Y_base)

for P, K in zip(P_values, K_values):
    # Slice the first K columns (most recent K lags)
    midas_data[K] = (Y_base, X_base[:, :K])

print(f"Base dataset: T={T} quarters, K_max={K_max} monthly lags")
print(f"Lag specifications: K = {K_values}")

In [ ]:
# --- Compute AIC/BIC for each lag specification ---

def compute_ic(sse, T, k):
    """AIC and BIC from SSE, sample size T, and parameter count k."""
    aic = T * np.log(sse / T) + 2 * k
    bic = T * np.log(sse / T) + k * np.log(T)
    return aic, bic


print("Estimating Beta MIDAS for each lag order (this takes ~30 seconds)...")
results_ic = []

for P, K in zip(P_values, K_values):
    Y, X = midas_data[K]
    est = estimate_midas(Y, X, beta_weights)

    k_restricted = 4   # alpha, beta, theta1, theta2
    aic, bic = compute_ic(est['sse'], T, k_restricted)
    r2 = 1 - est['sse'] / np.sum((Y - Y.mean())**2)

    results_ic.append({
        'P': P, 'K': K, 'k_params': k_restricted,
        'theta1': est['theta1'], 'theta2': est['theta2'],
        'sse': est['sse'], 'r2': r2,
        'aic': aic, 'bic': bic
    })
    print(f"  P={P} (K={K:2d}): SSE={est['sse']:.4f}, AIC={aic:.2f}, BIC={bic:.2f}, "
          f"theta=({est['theta1']:.2f},{est['theta2']:.2f})")

print("\nDone.")

In [ ]:
# --- Display the model selection table ---

df_ic = pd.DataFrame(results_ic)
bic_min = df_ic['bic'].min()
aic_min = df_ic['aic'].min()

print("Model Selection Table — Beta MIDAS, varying lag order")
print("=" * 75)
print(f"{'P':>4} {'K':>4} {'k':>4} {'SSE':>8} {'R²':>6} {'AIC':>9} {'BIC':>9}  Notes")
print("-" * 75)

for row in results_ic:
    notes = []
    if abs(row['bic'] - bic_min) < 0.01:
        notes.append("<- BIC min")
    if abs(row['aic'] - aic_min) < 0.01:
        notes.append("<- AIC min")
    note_str = "  ".join(notes)
    print(f"{row['P']:>4} {row['K']:>4} {row['k_params']:>4} "
          f"{row['sse']:>8.4f} {row['r2']:>6.3f} "
          f"{row['aic']:>9.2f} {row['bic']:>9.2f}  {note_str}")

print("-" * 75)
print(f"\nNote: k=4 for ALL restricted MIDAS specifications (alpha, beta, theta1, theta2).")
print(f"      BIC penalty = 4 * ln({T}) = {4 * np.log(T):.2f} for all rows.")

P_star = df_ic.loc[df_ic['bic'].idxmin(), 'P']
K_star = df_ic.loc[df_ic['bic'].idxmin(), 'K']
print(f"\nBIC-selected model: P={P_star} quarterly lags (K={K_star} monthly lags)")

### Interpretation

- **SSE decreases as K grows**: More lags give the weight function more history to draw on.
- **AIC/BIC penalty stays at $k \ln T$ / $2k$** for all rows: The restricted Beta MIDAS still has only 4 free parameters.
- **BIC identifies the inflection point**: Beyond the BIC minimum, the SSE improvement is negligible while the penalty remains constant.

This is fundamentally different from U-MIDAS where adding more lags also adds more parameters — BIC would penalize U-MIDAS heavily for large K.

In [ ]:
section_divider("4. Weight Function Family Comparison")

## 4. Weight Function Family Comparison

With the lag order selected (K = K*), we now compare:
1. OLS-aggregate (equal weights, k=2)
2. Beta MIDAS (k=4)
3. Almon MIDAS (k=4)
4. U-MIDAS (k=K+1, only feasible for small K)

In [ ]:
# Use K=9 for weight family comparison (feasible for U-MIDAS)
K_compare = 9
Y, X = midas_data[K_compare]

print(f"Comparing weight families at K={K_compare} monthly lags (P=3 quarterly lags)")
print("Estimating models...")

# --- OLS-aggregate (equal weights) ---
w_equal = np.ones(K_compare) / K_compare
xw_equal = X @ w_equal
lr_ols = LinearRegression().fit(xw_equal.reshape(-1, 1), Y)
sse_ols = np.sum((Y - lr_ols.predict(xw_equal.reshape(-1, 1)))**2)
aic_ols, bic_ols = compute_ic(sse_ols, T, 2)  # alpha, beta only
r2_ols = 1 - sse_ols / np.sum((Y - Y.mean())**2)

# --- Beta MIDAS ---
est_beta = estimate_midas(Y, X, beta_weights)
aic_beta, bic_beta = compute_ic(est_beta['sse'], T, 4)
r2_beta = 1 - est_beta['sse'] / np.sum((Y - Y.mean())**2)

# --- Almon MIDAS ---
est_almon = estimate_midas(
    Y, X, almon_weights,
    starts=[(-0.1, -0.01), (0.0, -0.05), (0.1, -0.02), (-0.2, -0.01)]
)
aic_almon, bic_almon = compute_ic(est_almon['sse'], T, 4)
r2_almon = 1 - est_almon['sse'] / np.sum((Y - Y.mean())**2)

# --- U-MIDAS (OLS on all K lags) ---
lr_u = LinearRegression().fit(X, Y)
sse_u = np.sum((Y - lr_u.predict(X))**2)
k_u = K_compare + 1  # intercept + K coefficients
aic_u, bic_u = compute_ic(sse_u, T, k_u)
r2_u = 1 - sse_u / np.sum((Y - Y.mean())**2)

print("Done.")

In [ ]:
# Display weight family comparison table
models = [
    ('OLS-aggregate', 2,        sse_ols,        aic_ols,        bic_ols,        r2_ols),
    ('Beta MIDAS',   4,        est_beta['sse'], aic_beta,       bic_beta,       r2_beta),
    ('Almon MIDAS',  4,        est_almon['sse'],aic_almon,      bic_almon,      r2_almon),
    (f'U-MIDAS',     k_u,      sse_u,          aic_u,          bic_u,          r2_u),
]

bic_vals = [m[4] for m in models]
bic_winner = models[np.argmin(bic_vals)][0]

print(f"Weight Function Comparison (K={K_compare}, T={T})")
print("=" * 72)
print(f"{'Model':<18} {'k':>4} {'SSE':>8} {'R²':>6} {'AIC':>9} {'BIC':>9}  Notes")
print("-" * 72)

for name, k, sse, aic, bic, r2 in models:
    note = "  <- BIC winner" if name == bic_winner else ""
    if name.startswith('U-MIDAS'):
        note += f"  (k={k} >> 4)"
    print(f"{name:<18} {k:>4} {sse:>8.4f} {r2:>6.3f} {aic:>9.2f} {bic:>9.2f}{note}")

print("-" * 72)
print(f"\nBIC winner: {bic_winner}")
print(f"\nNote: U-MIDAS has k={k_u} parameters vs k=4 for restricted MIDAS.")
print(f"      BIC penalty difference: {(k_u-4)*np.log(T):.2f} extra units.")

In [ ]:
# --- Visualize the estimated weight functions ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
j = np.arange(K_compare)

# Left panel: weight function shapes
ax = axes[0]
ax.bar(j - 0.2, w_equal, width=0.18, alpha=0.75, label='OLS-aggregate (equal)', color='gray')
ax.bar(j,       est_beta['weights'], width=0.18, alpha=0.85,
       label=f"Beta (θ₁={est_beta['theta1']:.2f}, θ₂={est_beta['theta2']:.2f})", color='steelblue')
ax.bar(j + 0.2, est_almon['weights'], width=0.18, alpha=0.85,
       label=f"Almon (θ₁={est_almon['theta1']:.3f}, θ₂={est_almon['theta2']:.4f})", color='tomato')

ax.axhline(1/K_compare, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Equal weight')
ax.set_xlabel('Lag j (j=0 is most recent month)')
ax.set_ylabel('Weight')
ax.set_title(f'Estimated Weight Functions (K={K_compare})')
ax.legend(fontsize=8)
ax.set_xticks(j)
ax.set_xticklabels([f'j={jj}' for jj in j], fontsize=7)

# Quarter annotation
for qstart, qlab in [(0, 'Q cur'), (3, 'Q-1'), (6, 'Q-2')]:
    ax.axvspan(qstart - 0.4, qstart + 2.4, alpha=0.04, color='blue')
    ax.text(qstart + 1, ax.get_ylim()[1] * 0.95, qlab, ha='center', fontsize=7, color='navy')

# Right panel: U-MIDAS unrestricted weights
ax2 = axes[1]
beta_u_vals = lr_u.coef_
ax2.bar(j - 0.15, est_beta['weights'], width=0.28, alpha=0.8, label='Beta MIDAS (4 params)', color='steelblue')
ax2.bar(j + 0.15, beta_u_vals / beta_u_vals.sum(),
        width=0.28, alpha=0.8,
        label=f'U-MIDAS ({k_u} params, normalized)', color='orange')
ax2.axhline(1/K_compare, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax2.set_xlabel('Lag j (j=0 is most recent month)')
ax2.set_ylabel('Normalized coefficient')
ax2.set_title('Beta MIDAS vs. U-MIDAS Coefficients')
ax2.legend(fontsize=9)
ax2.set_xticks(j)
ax2.set_xticklabels([f'j={jj}' for jj in j], fontsize=7)

plt.suptitle('Weight Function Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('weight_family_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
section_divider("5. Expanding-Window Cross-Validation")

## 5. Expanding-Window Cross-Validation

AIC/BIC measures in-sample fit. The expanding-window protocol measures **out-of-sample** predictive accuracy — the gold standard for forecasting model evaluation.

```
Q1...Q50 | Q51  <- estimate on first 50, predict Q51
Q1...Q51 | Q52  <- re-estimate on first 51, predict Q52
...
Q1...QT-1 | QT  <- re-estimate on T-1, predict QT
```

RMSE = $\sqrt{\frac{1}{T - T_0} \sum_{t=T_0}^{T-1} (y_{t+1} - \hat{y}_{t+1|t})^2}$

In [ ]:
def expanding_window_cv(Y, X_dict, min_train=30):
    """
    Expanding-window one-step-ahead cross-validation for MIDAS models.

    Parameters
    ----------
    Y : np.ndarray, shape (T,)
    X_dict : dict
        Keys are model names, values are (X, weight_fn) tuples.
        weight_fn=None signals U-MIDAS (OLS).
    min_train : int
        Minimum training size.

    Returns
    -------
    rmse_dict : dict, str -> float
    errors_dict : dict, str -> list of squared errors
    """
    T = len(Y)
    errors_dict = {name: [] for name in X_dict}

    for end in range(min_train, T):  # Train on Y[:end], predict Y[end]
        Y_train = Y[:end]
        y_test  = Y[end]

        for name, (X, weight_fn) in X_dict.items():
            X_train = X[:end]
            x_test  = X[end]

            if weight_fn is None:
                # U-MIDAS: OLS on all K lags
                model = LinearRegression().fit(X_train, Y_train)
                y_hat = model.predict(x_test.reshape(1, -1))[0]
            else:
                # Restricted MIDAS: profile NLS
                est = estimate_midas(Y_train, X_train, weight_fn,
                                     starts=[(1.0, 5.0), (1.5, 4.0)])
                w = weight_fn(X_train.shape[1], est['theta1'], est['theta2'])
                xw_test = float(x_test @ w)
                y_hat = est['alpha'] + est['beta'] * xw_test

            errors_dict[name].append((y_test - y_hat)**2)

    rmse_dict = {name: np.sqrt(np.mean(errs)) for name, errs in errors_dict.items()}
    return rmse_dict, errors_dict


print("Running expanding-window CV (may take 1-2 minutes)...")
print("(Models re-estimated at each step — this is the correct evaluation protocol)")

Y_cv, X_cv9 = midas_data[9]
_, X_cv12   = midas_data[12]

cv_models = {
    'OLS-aggregate (K=9)':  (X_cv9,  None),          # weight_fn=None -> OLS on equal-weight
    'Beta MIDAS (K=9)':     (X_cv9,  beta_weights),
    'Almon MIDAS (K=9)':    (X_cv9,  almon_weights),
    'Beta MIDAS (K=12)':    (X_cv12, beta_weights),
}

# OLS-aggregate uses pre-averaged x — handle separately
K9 = 9
w_eq9 = np.ones(K9) / K9
X_ols9 = (X_cv9 @ w_eq9).reshape(-1, 1)

# Compute OLS-aggregate manually
MIN_TRAIN = 30
ols_errors = []
for end in range(MIN_TRAIN, T):
    Xtr = X_ols9[:end]
    lr = LinearRegression().fit(Xtr, Y_cv[:end])
    y_hat = lr.predict(X_ols9[end].reshape(1, -1))[0]
    ols_errors.append((Y_cv[end] - y_hat)**2)
rmse_ols = np.sqrt(np.mean(ols_errors))

# Compute restricted MIDAS models
rmse_dict, errors_dict = expanding_window_cv(
    Y_cv,
    {
        'Beta MIDAS (K=9)':  (X_cv9,  beta_weights),
        'Almon MIDAS (K=9)': (X_cv9,  almon_weights),
        'Beta MIDAS (K=12)': (X_cv12, beta_weights),
    },
    min_train=MIN_TRAIN
)

rmse_dict['OLS-aggregate (K=9)'] = rmse_ols

print("\nExpanding-window RMSE Results:")
print(f"{'Model':<22} {'OOS RMSE':>10}")
print("-" * 35)
for name, rmse in sorted(rmse_dict.items(), key=lambda x: x[1]):
    print(f"{name:<22} {rmse:>10.4f}")

In [ ]:
# --- Plot cumulative squared errors over time ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: RMSE bar chart
ax = axes[0]
sorted_models = sorted(rmse_dict.items(), key=lambda x: x[1])
names = [m[0] for m in sorted_models]
rmses = [m[1] for m in sorted_models]
colors = ['#d62728' if 'OLS' in n else '#1f77b4' if 'Beta' in n else '#ff7f0e'
          for n in names]
bars = ax.barh(names, rmses, color=colors, alpha=0.8)
ax.set_xlabel('Out-of-Sample RMSE')
ax.set_title('Expanding-Window CV: RMSE Comparison')
for bar, rmse in zip(bars, rmses):
    ax.text(rmse + 0.002, bar.get_y() + bar.get_height()/2,
            f'{rmse:.4f}', va='center', fontsize=9)
ax.set_xlim(0, max(rmses) * 1.2)

# Right: Cumulative squared error over the holdout period
ax2 = axes[1]
n_oos = T - MIN_TRAIN
time_axis = np.arange(n_oos)

# Plot OLS-aggregate
ax2.plot(time_axis, np.cumsum(ols_errors), color='#d62728', linewidth=2,
         label='OLS-aggregate', linestyle='--')

# Plot MIDAS models
line_styles = ['-', ':', '-.']
model_colors = {'Beta MIDAS (K=9)': '#1f77b4', 'Almon MIDAS (K=9)': '#ff7f0e',
                'Beta MIDAS (K=12)': '#2ca02c'}
for (name, errs), ls in zip(errors_dict.items(), line_styles):
    ax2.plot(time_axis[:len(errs)], np.cumsum(errs),
             color=model_colors.get(name, 'gray'), linewidth=2,
             label=name, linestyle=ls)

ax2.set_xlabel(f'Out-of-sample period (starting at observation {MIN_TRAIN})')
ax2.set_ylabel('Cumulative Squared Error')
ax2.set_title('Cumulative Forecast Error Over Time')
ax2.legend(fontsize=8)

plt.suptitle('Out-of-Sample Forecast Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('oos_forecast_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 6. Ljung-Box Test for Residual Autocorrelation

Before finalizing a model, check whether residuals have serial correlation. If they do, an AR term is needed (MIDAS-AR).

$$Q_{LB}(p) = T(T+2) \sum_{k=1}^{p} \frac{\hat{\rho}_k^2}{T-k} \sim \chi^2_p \text{ under } H_0$$

Reject if $p\text{-value} < 0.10$.

In [ ]:
def ljung_box_test(residuals, n_lags=4):
    """
    Ljung-Box portmanteau test for serial correlation in residuals.

    H0: No autocorrelation up to lag n_lags.
    Reject at 10% -> consider adding AR terms.

    Returns: Q statistic, p-value, list of autocorrelations.
    """
    T = len(residuals)
    acf_vals = []
    for k in range(1, n_lags + 1):
        acf_k = np.corrcoef(residuals[k:], residuals[:-k])[0, 1]
        acf_vals.append(acf_k)
    acf_vals = np.array(acf_vals)

    Q = T * (T + 2) * np.sum(acf_vals**2 / (T - np.arange(1, n_lags + 1)))
    p_val = 1 - chi2.cdf(Q, df=n_lags)

    print(f"Ljung-Box Q({n_lags}) = {Q:.3f}, p-value = {p_val:.4f}")
    for k, rho in enumerate(acf_vals, 1):
        print(f"  rho_{k} = {rho:+.4f}")
    if p_val < 0.10:
        print(f"  -> Reject H0 at 10%: serial correlation detected. Add AR term.")
    else:
        print(f"  -> Fail to reject H0: no significant serial correlation.")

    return Q, p_val, acf_vals


# Estimate Beta MIDAS on full K=12 dataset
Y_full, X_full = midas_data[K_star]
est_final = estimate_midas(Y_full, X_full, beta_weights)
residuals_final = est_final['residuals']

print(f"Ljung-Box Test on Residuals — Beta MIDAS (K={K_star}, T={T})")
print("=" * 55)
Q, p_val, acf_vals = ljung_box_test(residuals_final, n_lags=4)

In [ ]:
# --- Residual diagnostics panel ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Residuals over time
ax = axes[0]
ax.plot(residuals_final, color='steelblue', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(range(len(residuals_final)), residuals_final, 0,
                where=(residuals_final > 0), alpha=0.3, color='steelblue')
ax.fill_between(range(len(residuals_final)), residuals_final, 0,
                where=(residuals_final < 0), alpha=0.3, color='tomato')
ax.set_title(f'Residuals over Time (K={K_star})')
ax.set_xlabel('Observation')
ax.set_ylabel('Residual')

# Residual histogram
ax2 = axes[1]
ax2.hist(residuals_final, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
ax2.set_title('Residual Distribution')
ax2.set_xlabel('Residual')
ax2.set_ylabel('Frequency')
# Normal overlay
from scipy.stats import norm
xr = np.linspace(residuals_final.min(), residuals_final.max(), 100)
ax2.plot(xr, norm.pdf(xr, residuals_final.mean(), residuals_final.std()) * len(residuals_final) *
         (residuals_final.max() - residuals_final.min()) / 15,
         color='tomato', linewidth=2, label='Normal fit')
ax2.legend(fontsize=9)

# ACF bar chart
ax3 = axes[2]
lags = np.arange(1, len(acf_vals) + 1)
colors_acf = ['tomato' if abs(r) > 1.96/np.sqrt(T) else 'steelblue' for r in acf_vals]
ax3.bar(lags, acf_vals, color=colors_acf, alpha=0.8)
ci = 1.96 / np.sqrt(T)
ax3.axhline(ci,  color='gray', linestyle='--', linewidth=1, label=f'±1.96/√T')
ax3.axhline(-ci, color='gray', linestyle='--', linewidth=1)
ax3.axhline(0,   color='black', linewidth=0.5)
ax3.set_xlabel('Lag')
ax3.set_ylabel('Autocorrelation')
ax3.set_title(f'Residual ACF (Q={Q:.2f}, p={p_val:.3f})')
ax3.legend(fontsize=8)
ax3.set_xticks(lags)

plt.suptitle(f'Residual Diagnostics — Beta MIDAS (K={K_star})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('residual_diagnostics_model_selection.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 7. Summary: Final Model Selection Table

Combining BIC, OOS RMSE, and residual diagnostics into a single recommendation table.

In [ ]:
# Collect summary statistics for the full selection table
# Using K=9 for U-MIDAS comparison (feasible with T~100), K=K_star for primary

Y9, X9 = midas_data[9]
est_beta9  = estimate_midas(Y9, X9, beta_weights)
est_almon9 = estimate_midas(Y9, X9, almon_weights,
                            starts=[(-0.1, -0.01), (0.0, -0.05), (0.1, -0.02)])

lr_ols9 = LinearRegression().fit((X9 @ np.ones(9)/9).reshape(-1,1), Y9)
sse_ols9 = np.sum((Y9 - lr_ols9.predict((X9 @ np.ones(9)/9).reshape(-1,1)))**2)

lr_u9 = LinearRegression().fit(X9, Y9)
sse_u9 = np.sum((Y9 - lr_u9.predict(X9))**2)

# Get OOS RMSE from the earlier CV run (K=9, K=12 Beta)
rmse_beta9  = rmse_dict.get('Beta MIDAS (K=9)', np.nan)
rmse_almon9 = rmse_dict.get('Almon MIDAS (K=9)', np.nan)
rmse_ols9   = rmse_dict.get('OLS-aggregate (K=9)', np.nan)
rmse_beta12 = rmse_dict.get('Beta MIDAS (K=12)', np.nan)

final_rows = [
    ('OLS-aggregate (K=9)',  2,  sse_ols9,       *compute_ic(sse_ols9, T, 2),       rmse_ols9),
    ('Beta MIDAS (K=9)',     4,  est_beta9['sse'],*compute_ic(est_beta9['sse'], T, 4), rmse_beta9),
    ('Almon MIDAS (K=9)',    4,  est_almon9['sse'],*compute_ic(est_almon9['sse'], T,4), rmse_almon9),
    (f'U-MIDAS (K=9)',       10, sse_u9,         *compute_ic(sse_u9, T, 10),         np.nan),
    (f'Beta MIDAS (K={K_star})', 4, est_final['sse'], *compute_ic(est_final['sse'], T, 4), rmse_beta12),
]

bic_final = [r[4] for r in final_rows]
oos_final = [r[5] for r in final_rows]
bic_winner_idx = np.argmin(bic_final)
oos_winner_idx = int(np.nanargmin(oos_final))

print(f"Final Model Selection Table (T={T})")
print("=" * 85)
print(f"{'Model':<25} {'k':>3} {'SSE':>8} {'AIC':>9} {'BIC':>9} {'OOS RMSE':>10}  Notes")
print("-" * 85)
for i, (name, k, sse, aic, bic, oos_rmse) in enumerate(final_rows):
    notes = []
    if i == bic_winner_idx: notes.append("BIC win")
    if i == oos_winner_idx: notes.append("OOS win")
    oos_str = f"{oos_rmse:>10.4f}" if not np.isnan(oos_rmse) else f"{'N/A':>10}"
    print(f"{name:<25} {k:>3} {sse:>8.4f} {aic:>9.2f} {bic:>9.2f} {oos_str}  {', '.join(notes)}")
print("-" * 85)
print(f"\nRecommendation: Beta MIDAS with K={K_star} lags (P={K_star//3} quarterly lags).")
print(f"  Rationale: consistent BIC and OOS RMSE performance.")

## 8. Self-Check

Run the cells below to verify your understanding. Each assertion tests a key concept from this notebook.

In [ ]:
print("Self-check assertions...")
passed = 0
total  = 6

# 1. Restricted MIDAS has k=4 regardless of K
k_values_seen = [r['k_params'] for r in results_ic]
assert all(k == 4 for k in k_values_seen), \
    f"FAIL 1: Expected k=4 for all restricted MIDAS, got {k_values_seen}"
passed += 1
print("  PASS 1: Restricted MIDAS has k=4 parameters for all K")

# 2. SSE decreases (or stays flat) as K increases
sse_vals = [r['sse'] for r in results_ic]
sse_diffs = np.diff(sse_vals)
assert all(d <= 0.5 for d in sse_diffs), \
    f"FAIL 2: SSE should not increase substantially with K, diffs={sse_diffs}"
passed += 1
print("  PASS 2: SSE decreases (or plateaus) as K grows")

# 3. Beta weights sum to 1
w_test = beta_weights(12, 1.5, 4.0)
assert abs(w_test.sum() - 1.0) < 1e-10, \
    f"FAIL 3: Beta weights must sum to 1, got {w_test.sum()}"
passed += 1
print("  PASS 3: Beta weights sum to 1.0")

# 4. Almon weights sum to 1
w_almon_test = almon_weights(9, -0.1, -0.01)
assert abs(w_almon_test.sum() - 1.0) < 1e-10, \
    f"FAIL 4: Almon weights must sum to 1, got {w_almon_test.sum()}"
passed += 1
print("  PASS 4: Almon weights sum to 1.0")

# 5. U-MIDAS has more parameters than restricted MIDAS
assert k_u > 4, \
    f"FAIL 5: U-MIDAS (k={k_u}) should have more params than restricted MIDAS (k=4)"
passed += 1
print(f"  PASS 5: U-MIDAS has k={k_u} > 4 parameters")

# 6. Ljung-Box returns a valid chi-squared p-value
assert 0.0 <= p_val <= 1.0, \
    f"FAIL 6: p-value must be in [0,1], got {p_val}"
passed += 1
print(f"  PASS 6: Ljung-Box p-value = {p_val:.4f} (valid)")

print(f"\n{'='*40}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")
else:
    print("Some checks failed — review the relevant sections above.")

## Summary

### What We Covered

| Topic | Key Takeaway |
|-------|-------------|
| AIC/BIC for MIDAS | k=4 for restricted MIDAS regardless of K; SSE drives selection |
| Lag order selection | BIC identifies the K where SSE improvement plateaus |
| Weight family comparison | Beta and Almon similar; U-MIDAS penalized heavily by BIC |
| Expanding-window CV | OOS RMSE validates BIC selection; Beta MIDAS typically wins |
| Ljung-Box test | Check residuals before finalizing; add AR if p < 0.10 |

### Decision Rule Summary

1. Start with Beta MIDAS, P=4 (K=12) as the default specification
2. Use BIC to check whether P=2 or P=6 improves on P=4
3. Compare Beta vs. Almon by BIC (usually very similar)
4. Run Ljung-Box Q(4) on residuals — if p < 0.10, add AR(1)
5. Validate final model with expanding-window RMSE

### Next

**Notebook 03:** Robust inference — HAC standard errors, hypothesis testing, and bootstrap confidence intervals for the selected model.

In [ ]:
key_takeaways(["Robust inference — HAC standard errors, hypothesis testing, and bootstrap confid"])